# Lab 29: The Lip-Reading Override

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-29.ipynb)

**What you will do:** watch the McGurk video with your eyes open and then closed, log what syllable you and a few friends actually hear, compare your fusion rate with the range the book reports, and test whether an open speech-recognition model can be pulled off course by a misleading text hint the way your own perception is pulled by the mismatched video.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 29 you watched a video of a face saying "ga" while the soundtrack played "ba." With
your eyes open, most people hear neither of these: they hear "da," a fused syllable that no
one actually produced. Closing your eyes brings back the true auditory signal, "ba," and
opening them again brings "da" straight back. The fusion is automatic — you cannot switch it
off just by knowing the trick.

## Record your results

Log what each viewer heard, once with eyes open and once with eyes closed, using an
anonymous label rather than a real name if you share this further. "Fused" below means the
eyes-open answer was neither the seen "ga" nor the heard "ba" — typically "da" or "tha."

In [ ]:
# example data: replace with your own and your friends' reports
trials = pd.DataFrame({
    "viewer": ["you", "you", "friend A", "friend A", "friend B", "friend B", "friend C", "friend C"],
    "condition": ["eyes open", "eyes closed", "eyes open", "eyes closed",
                  "eyes open", "eyes closed", "eyes open", "eyes closed"],
    "heard": ["da", "ba", "da", "ba", "tha", "ba", "ba", "ba"],
})
display(trials)

open_trials = trials[trials.condition == "eyes open"]
fused = open_trials.heard.isin(["da", "tha"])  # neither the seen "ga" nor the heard "ba"
print(f"Fused with eyes open: {fused.sum()} / {len(open_trials)} viewers ({fused.mean():.0%})")

trials.groupby(["condition", "heard"]).size().unstack(fill_value=0).plot.bar(figsize=(6, 3))
plt.ylabel("number of viewers"); plt.xticks(rotation=0)
plt.title("What viewers heard, by condition"); plt.show()

## Compare

Lab 29 does not give one fixed fusion rate. It reports that some studies find fusion in
nearly everyone, others find it in only about half of participants, and a minority of
viewers fuse rarely or never — even watching the identical video. Sort your own group's
eyes-open results into one of these three rough bands.

In [ ]:
rate = fused.mean()
if rate >= 0.8:
    band = "near ceiling -- close to the high end of the range Lab 29 reports"
elif rate >= 0.35:
    band = "around half -- close to the middle of the range Lab 29 reports"
else:
    band = ("a minority -- Lab 29 says some people show little or no fusion, and this is a "
             "genuine, well-documented exception, not a sign they answered incorrectly")
print(f"Your group's eyes-open fusion rate: {rate:.0%}")
print(band)

## The AI side

Whisper, an open speech-recognition model (Radford et al., 2023), can also be pushed off
course by a second signal that conflicts with what it hears — not a video this time, but a
short piece of text handed to it beforehand as a "prompt." Below, the same noisy audio clip
is transcribed three times: with no prompt, with a prompt naming words that really are in
the clip, and with a prompt naming unrelated words. Watch whether the misleading prompt
pulls the transcription towards its own wording, even though the audio never changes — the
same shape of effect as the mismatched video pulling your percept towards "da."

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers datasets soundfile

import numpy as np
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

# A short, real spoken sentence -- the standard tiny test clip used in Whisper tutorials
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
audio = ds[0]["audio"]["array"]
sr = ds[0]["audio"]["sampling_rate"]
print("What was actually said:", ds[0]["text"])

# Degrade the clip so the audio alone is genuinely ambiguous -- the audio equivalent of
# dubbing "ba" onto a video of "ga"
rng = np.random.default_rng(0)
noise = rng.normal(0, 1, len(audio))
target_snr_db = -3
scale = np.sqrt(np.mean(audio ** 2) / (np.mean(noise ** 2) * 10 ** (target_snr_db / 10)))
noisy_audio = audio + scale * noise

def transcribe(audio_array, prompt=None):
    inputs = processor(audio_array, sampling_rate=sr, return_tensors="pt")
    prompt_ids = processor.get_prompt_ids(prompt, return_tensors="pt") if prompt else None
    ids = model.generate(inputs.input_features, prompt_ids=prompt_ids)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

results = pd.DataFrame({
    "condition": ["no prompt", "correct-content prompt", "misleading prompt"],
    "Whisper's transcription": [
        transcribe(noisy_audio),
        transcribe(noisy_audio, prompt="quilter apostle middle classes"),
        transcribe(noisy_audio, prompt="weather forecast for tomorrow afternoon"),
    ],
})
display(results)

A correct-content prompt should make the noisy clip easier to transcribe accurately — the
same way knowing a sentence makes sine-wave speech (Lab 31) suddenly clear. A misleading
prompt can pull individual words towards its own vocabulary even though the sound never
changed, echoing how the sight of "ga" pulls your percept away from the "ba" you actually
heard. The analogy has limits: Whisper's "prompt" is a short piece of text you choose to
supply, not a second sense organ it perceives unprompted, and there is no felt experience of
one output winning over another — it is only assigning different probabilities to
sequences of words. Results can also vary with how much noise was added and with the exact
wording of the prompts.

## Pool the class data

Pool syllable reports across the class or a larger friend group: each person adds one row
(an anonymous ID, never a name, plus what they heard with eyes open and with eyes closed).
The cell below computes the eyes-open fusion rate with a bootstrap interval, so you can see
how the estimate settles down once more than a handful of people have tried it.

In [ ]:
import io
csv_text = """id,heard_eyes_open,heard_eyes_closed
P01,da,ba
P02,da,ba
P03,tha,ba
P04,ba,ba
P05,da,ba
P06,da,ba
P07,ga,ba
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
df["fused"] = df.heard_eyes_open.isin(["da", "tha"])

boot = [df.fused.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class fusion rate (eyes open): {df.fused.mean():.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(df)})")

df.heard_eyes_open.value_counts().plot.bar(color="steelblue", figsize=(5, 3))
plt.ylabel("number of people"); plt.title("What the class heard, eyes open"); plt.show()

## Questions to think about

1. Lab 29 explains the specific fused percept "da" through reliability weighting — vision rules out "ba" (the mouth clearly is not closing for a bilabial), and audition breaks the remaining tie between "da" and "ga." Given that account, what result would you predict if the audio track played "ga" while the video showed a mouth clearly saying "ba"? Try to find or imagine this reversed pairing.
2. In the AI Lab, did the misleading prompt actually change any words, or did it leave the transcription alone? What does either outcome tell you about how strongly Whisper weighs a text prompt against a fairly clear audio signal, compared with how strongly your brain weighs a very clear visual signal against a clear auditory one?
3. The book reports that the McGurk effect is on average weaker in Japanese listeners than in English listeners hearing the same syllables. If you tested people who are highly fluent in more than one language, what would a predictive-processing account expect, and why?
4. You cannot un-hear "da" once you know the trick. Is there a comparable case in the AI Lab where Whisper's output, once generated, is easier or harder to "unbias" than your own perception is?

## Challenge

Show the video to people from different first-language backgrounds if you can, without
explaining the trick first, and log what each person hears with eyes open. Note whether the
pattern in your small sample leans towards the near-universal fusion the book reports for many
English speakers, or towards the weaker fusion it reports for Japanese listeners.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-29.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")